In [1]:
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Lambda, Add
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np

# Load your pretrained models
model_1 = load_model('/kaggle/input/mobilenetv3small/keras/default/1/my_keras_model.keras')
model_2 = load_model('/kaggle/input/vgg-16/keras/default/1/my_keras_model.keras')
model_3 = load_model('/kaggle/input/densenet121/keras/default/1/DenseNet121_model.keras')
model_4 = load_model('/path/to/Xception.h5')
model_5 = load_model('/kaggle/input/resnet50/keras/default/1/ResNet50_model.keras')
model_6 = load_model('/path/to/AlexNet.h5')
model_7 = load_model('/path/to/InceptionV3.h5')


# List of models
models = [model_1, model_2, model_3, model_4, model_5, model_6, model_7]

# Define weights for each model (weights must sum to 1 for normalization)
weights = [0.1, 0.15, 0.2, 0.1, 0.15, 0.1, 0.1, 0.1]

# Input layer
model_input = Input(shape=(224, 224, 3))

# Get weighted outputs from each model
weighted_outputs = [Lambda(lambda x: w * x)(model(model_input)) for model, w in zip(models, weights)]

# Sum the weighted outputs
ensemble_output = Add()(weighted_outputs)

# Final weighted ensemble model
ensemble_model = Model(inputs=model_input, outputs=ensemble_output, name='weighted_ensemble')

# Compile the ensemble model
sgd = SGD(learning_rate=0.001, momentum=0.9, nesterov=True)
ensemble_model.compile(optimizer=sgd, loss='categorical_crossentropy', metrics=['accuracy'])

# Define the image data generators
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=30, width_shift_range=0.2, height_shift_range=0.2, shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load datasets
train_generator = train_datagen.flow_from_directory(
    '/kaggle/input/rice-leaf-split-dataset/kaggle/working/split-dataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

validation_generator = val_datagen.flow_from_directory(
    '/kaggle/input/rice-leaf-split-dataset/kaggle/working/split-dataset/val',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    '/kaggle/input/rice-leaf-split-dataset/kaggle/working/split-dataset/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Define callbacks
early_stopping = EarlyStopping(monitor="val_loss", patience=3, verbose=1)
model_checkpoint = ModelCheckpoint(
    '/path/to/save_best_model.keras', save_best_only=True, monitor="val_loss", mode='min'
)

# Train the ensemble model
history = ensemble_model.fit(
    train_generator,
    epochs=20,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    callbacks=[early_stopping, model_checkpoint]
)

# Save the final ensemble model
ensemble_model.save('/path/to/final_weighted_ensemble_model.keras')

# Save the training history
history_path = '/path/to/weighted_ensemble_history.pkl'
with open(history_path, 'wb') as file:
    pickle.dump(history.history, file)

print(f"Training complete. Final model saved at /path/to/final_weighted_ensemble_model.keras.")
print(f"Training history saved at {history_path}.")


KeyboardInterrupt: 